# Multiplex LNP spot detection — Round 2 (S3/S4)

This notebook reproduces full-barcode spot detection for the regions used in Figure 2 and Supplementary Figures 6–11. It begins with registered multichannel TIFF stacks, detects marker candidates, decodes LNP barcodes, assigns spots to segmented-cell centroids, writes analysis CSV tables, and displays a representative spot/codebook comparison in the notebook. The display cell does not save an image.

Regions not presented in the manuscript are excluded.


## Environment

Required packages: `numpy`, `pandas`, `matplotlib`, `tifffile`, `scipy`, and `scikit-image`. CuPy is optional; the notebook automatically uses the CPU when a compatible GPU installation is unavailable. Install packages in the environment before running the notebook.


In [ ]:
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import json
import logging
import sys
import time
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from scipy import ndimage as ndi
from scipy.spatial import cKDTree
from IPython.display import display


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-5s | %(message)s',
    datefmt='%H:%M:%S',
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger('multiplex-lnp-spot-detection')


def tic():
    return time.perf_counter()


def toc(t0, label=''):
    dt = time.perf_counter() - t0
    log.info(f'{label} done in {dt:.2f} s')
    return dt


def find_nanostamp_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidate = base / 'Manuscripts' / 'NanoSTAMP'
        if candidate.exists():
            return candidate
        if base.name == 'NanoSTAMP' and (base / 'Data').exists():
            return base
    raise FileNotFoundError(
        'Could not locate Manuscripts/NanoSTAMP. Start Jupyter inside the repository '
        'or set NANOSTAMP_ROOT below to the publication package directory.'
    )


NANOSTAMP_ROOT = find_nanostamp_root()
SEG_DIR = NANOSTAMP_ROOT / 'Data' / 'Figure_2_and_Supplementary_Figures_6_11_Multiplex_LNP' / 'Raw_Spot_Detection_Input' / 'Round_2_S3_S4'
OUTDIR = NANOSTAMP_ROOT / 'Data' / 'Figure_2_and_Supplementary_Figures_6_11_Multiplex_LNP' / 'Generated_Output' / 'Round_2_S3_S4'
OUTDIR.mkdir(parents=True, exist_ok=True)

REGIONS = [
    dict(region='S3_reg000', sample='registered_S3_reg000', features_name='20260708_registered_S3_reg000_integrated_registered_overlap_features.csv'),
    dict(region='S3_reg001', sample='registered_S3_reg001', features_name='20260708_registered_S3_reg001_integrated_registered_overlap_features.csv'),
    dict(region='S3_reg002', sample='registered_S3_reg002', features_name='20260708_registered_S3_reg002_integrated_registered_overlap_features.csv'),
    dict(region='S3_reg003', sample='registered_S3_reg003', features_name='20260708_registered_S3_reg003_integrated_registered_overlap_features.csv'),
    dict(region='S4_reg000', sample='registered_S4_reg000', features_name='20260708_registered_S4_reg000_integrated_registered_overlap_features.csv'),
    dict(region='S4_reg001', sample='registered_S4_reg001', features_name='20260708_registered_S4_reg001_integrated_registered_overlap_features.csv'),
    dict(region='S4_reg002', sample='registered_S4_reg002', features_name='20260708_registered_S4_reg002_integrated_registered_overlap_features.csv'),
    dict(region='S4_reg003', sample='registered_S4_reg003', features_name='20260708_registered_S4_reg003_integrated_registered_overlap_features.csv'),
]


@dataclass
class Params:
    log_sigma: float = 2.0
    log_sigmas: Tuple[float, ...] = (1.5, 2.0, 3.0, 5.0, 8.0, 10.0, 12.0, 15.0, 18.0, 20.0, 22.0, 25.0, 30.0)
    peak_width: int = 3
    nms_min_distance: float = 0.8
    threshold_peaks: float = 2.75
    initial_calib_threshold: float = 3.0
    use_raw_bright_rescue: bool = True
    raw_peak_percentile: float = 98.0
    use_raw_object_rescue: bool = True
    skip_raw_object_rescue: bool = False
    raw_object_percentile: float = 99.0
    raw_object_min_area: int = 4
    raw_object_max_area: int = 400
    raw_rescue_tile: int = 4096
    snr_core_radius: int = 1
    snr_annulus_inner: int = 3
    snr_annulus_outer: int = 6
    bright_snr_threshold: float = 1.75
    max_bright_markers: int = 10
    min_snr_margin: float = 0.02
    min_on_snr: float = 0.7
    expected_on_bits: Optional[int] = 6
    barcode_max_hamming_distance: int = 2
    hotpx_count_threshold: int = 2
    hotpx_sat_value: float = 65500.0
    cell_assignment_radius_px: float = 25.0
    min_spots_per_cell: int = 1
    min_spots_for_bit: int = 1
    roi_yx: Optional[Tuple[slice, slice]] = None


P = Params()
log.info(f'Input directory:  {SEG_DIR}')
log.info(f'Output directory: {OUTDIR}')
log.info(f'Parameters:       {P}')


## 1. Discover and validate registered input files


In [ ]:
def read_marker_list(path: Path) -> List[str]:
    return [line.strip() for line in path.read_text().splitlines() if line.strip()]


def open_stack(path: Path):
    """Open a TIFF stack as a memory-mapped (C, Y, X) array."""
    arr = tifffile.memmap(path)
    if arr.ndim == 2:
        arr = arr[np.newaxis, ...]
    if arr.ndim != 3:
        raise ValueError(f'Expected image stack as (C,Y,X), got {arr.shape} from {path}')
    return arr


def infer_integrated_shape(summary: dict, image_path: Path, n_markers: int) -> Tuple[int, int, int]:
    shape = summary.get('integrated_shape')
    if shape and all(v is not None for v in shape):
        return tuple(int(v) for v in shape)

    raw1 = summary.get('image1_raw_shape')
    raw2 = summary.get('image2_raw_shape')
    if raw1 and raw2 and len(raw1) >= 3 and len(raw2) >= 3:
        h = raw1[1]
        w = raw1[2]
        if h is not None and w is not None:
            return (int(n_markers), int(h), int(w))

    with tifffile.TiffFile(image_path) as tif:
        arr_shape = tif.series[0].shape
    if len(arr_shape) == 2:
        return (1, int(arr_shape[0]), int(arr_shape[1]))
    if len(arr_shape) == 3:
        return tuple(int(v) for v in arr_shape)
    raise ValueError(f'Could not infer integrated shape from summary or image: {image_path}')


def region_paths(item: dict) -> Dict[str, Path]:
    region = item['region']
    sample = item['sample']
    return {
        'image': SEG_DIR / f'{sample}_integrated_registered_overlap_crop.tif',
        'markerlist': SEG_DIR / f'{sample}_integrated_MarkerList.txt',
        'features': SEG_DIR / item.get('features_name', f'{region}_features.csv'),
        'tissue_mask': SEG_DIR / item.get('tissue_mask_name', f'{sample}_mask.npy'),
        'summary': SEG_DIR / f'{sample}_registration_summary.json',
    }


region_info = []
for item in REGIONS:
    paths = region_paths(item)
    missing = [name for name, path in paths.items() if name != 'tissue_mask' and not path.exists()]
    if missing:
        raise FileNotFoundError(f"{item['region']} missing: {missing}")

    markers = read_marker_list(paths['markerlist'])
    with open(paths['summary']) as f:
        summary = json.load(f)
    shape = infer_integrated_shape(summary, paths['image'], len(markers))
    barcode_markers = [m for m in markers if m.startswith('A')]
    region_info.append({**item, **paths, 'markers': markers, 'shape': shape, 'barcode_markers': barcode_markers})

pd.DataFrame([
    dict(region=r['region'], sample=r['sample'], image_shape=r['shape'], n_markers=len(r['markers']),
         barcode_markers=', '.join(r['barcode_markers']), features=r['features'].name)
    for r in region_info
])


## 2. Define the marker order and LNP barcode library


In [ ]:
# Keep the barcode bit order identical to v3 for direct comparison.
BARCODE_MARKERS = ['A6', 'A17', 'A55', 'A56', 'A76', 'A79', 'A20', 'A46', 'A28', 'A72', 'A2', 'A63', 'A126', 'A127', 'A128']

BARCODE_LIBRARY: Dict[str, str] = {
    '101001100011000': 'LNP_01',
    '100110100011000': 'LNP_02',
    '101010001011000': 'LNP_03',
    '000101101110000': 'LNP_04',
    '011010100101000': 'LNP_05',
    '010000110111000': 'LNP_06',
    '011110010100000': 'LNP_07',
    '111101010000000': 'LNP_08',
    '010100011011000': 'LNP_09',
    '011011000110000': 'LNP_10',
}


def validate_barcode_library():
    expected_len = len(BARCODE_MARKERS)
    names = list(BARCODE_LIBRARY.values())
    if len(names) != len(set(names)):
        raise ValueError('BARCODE_LIBRARY has duplicated LNP names; names should be unique.')
    for barcode, name in BARCODE_LIBRARY.items():
        if len(barcode) != expected_len:
            raise ValueError(
                f'{name} barcode {barcode} has length {len(barcode)}, '
                f'but BARCODE_MARKERS has length {expected_len}.'
            )
        bad = sorted(set(barcode) - {'0', '1'})
        if bad:
            raise ValueError(f'{name} barcode {barcode} contains non-binary symbols: {bad}')


def marker_sets_for_lnp_name(lnp_name: str) -> Tuple[List[str], List[str]]:
    matches = [barcode for barcode, name in BARCODE_LIBRARY.items() if name == lnp_name]
    if not matches:
        raise ValueError(f'{lnp_name!r} is not present in BARCODE_LIBRARY')
    barcode = matches[0]
    pos = [marker for marker, bit in zip(BARCODE_MARKERS, barcode) if bit == '1']
    neg = [marker for marker, bit in zip(BARCODE_MARKERS, barcode) if bit == '0']
    return pos, neg


def library_marker_positive_fraction() -> Dict[str, float]:
    fractions = {}
    for idx, marker in enumerate(BARCODE_MARKERS):
        vals = [int(barcode[idx]) for barcode in BARCODE_LIBRARY]
        fractions[marker] = float(np.mean(vals))
    return fractions


validate_barcode_library()
weights = sorted({sum(int(x) for x in barcode) for barcode in BARCODE_LIBRARY})
if len(weights) != 1:
    raise ValueError(f'BARCODE_LIBRARY must be constant-weight for this notebook; got weights {weights}')
EXPECTED_ON_BITS = int(weights[0])
P.expected_on_bits = EXPECTED_ON_BITS
LIBRARY_CODES = list(BARCODE_LIBRARY.keys())
LIBRARY_NAMES = [BARCODE_LIBRARY[k] for k in LIBRARY_CODES]
LIBRARY_MARKER_POSITIVE_FRACTION = library_marker_positive_fraction()
QC_POSITIVE_MARKERS = [m for m in BARCODE_MARKERS if LIBRARY_MARKER_POSITIVE_FRACTION[m] >= 0.5]
QC_NEGATIVE_MARKERS = [m for m in BARCODE_MARKERS if LIBRARY_MARKER_POSITIVE_FRACTION[m] < 0.5]

log.info(f'BARCODE_MARKERS ({len(BARCODE_MARKERS)}): {BARCODE_MARKERS}')
log.info(f'Barcode library entries: {len(BARCODE_LIBRARY)} -> {BARCODE_LIBRARY}')
log.info(f'EXPECTED_ON_BITS: {EXPECTED_ON_BITS}')
log.info(f'Library marker-positive fraction: {LIBRARY_MARKER_POSITIVE_FRACTION}')
log.info(f'QC positive markers: {QC_POSITIVE_MARKERS}')
log.info(f'QC negative markers: {QC_NEGATIVE_MARKERS}')


## 3. Define cell-level bit-count summaries


In [ ]:
def build_cell_bit_count_table(assigned_spots: pd.DataFrame, region_markers: List[str]) -> pd.DataFrame:
    cols = ['cell', *region_markers, 'decoded_spots', 'decoded_exact_spots', 'decoded_tolerant_spots',
            'dominant_barcode', 'dominant_barcode_count', 'dominant_lnp_call']
    if len(assigned_spots) == 0:
        return pd.DataFrame(columns=cols)
    assigned = assigned_spots[assigned_spots['cell'] > 0].copy()
    if len(assigned) == 0:
        return pd.DataFrame(columns=cols)

    def normalize_called_code(code: object) -> Optional[str]:
        # Cached tile checkpoints round-trip 'called_code' through CSV; pandas
        # infers an all-digit barcode column as int64 on read, stripping leading
        # zeros. Re-derive a clean zero-padded bit string from whatever came back.
        if pd.isna(code):
            return None
        if isinstance(code, (int, np.integer)):
            text = str(int(code))
        elif isinstance(code, float):
            if not float(code).is_integer():
                return None
            text = str(int(code))
        else:
            text = str(code).strip()
            if text.endswith('.0') and text[:-2].isdigit():
                text = text[:-2]
        if not text or any(ch not in '01' for ch in text):
            return None
        return text.zfill(len(region_markers))

    assigned['called_code'] = assigned['called_code'].map(normalize_called_code)
    assigned = assigned[assigned['called_code'].notna()].copy()
    if len(assigned) == 0:
        return pd.DataFrame(columns=cols)

    bit_matrix = np.array([[int(ch) for ch in code] for code in assigned['called_code']], dtype=int)
    bit_df = pd.DataFrame(bit_matrix, columns=region_markers, index=assigned.index)
    bit_df['cell'] = assigned['cell'].to_numpy(int)
    counts = bit_df.groupby('cell')[region_markers].sum().reset_index()

    summary = assigned.groupby('cell').agg(
        decoded_spots=('called_code', 'size'),
        decoded_exact_spots=('barcode_match_status', lambda s: int(np.sum(s == 'exact'))),
        decoded_tolerant_spots=('barcode_match_status', lambda s: int(np.sum(s == 'tolerant'))),
    ).reset_index()

    bc_counts = (assigned.groupby(['cell', 'called_code']).size().rename('n').reset_index()
                 .sort_values(['cell', 'n', 'called_code'], ascending=[True, False, True]))
    dominant_bc = bc_counts.drop_duplicates('cell').rename(columns={'called_code': 'dominant_barcode', 'n': 'dominant_barcode_count'})

    lnp_counts = (assigned.groupby(['cell', 'lnp_call']).size().rename('n').reset_index()
                  .sort_values(['cell', 'n', 'lnp_call'], ascending=[True, False, True]))
    dominant_lnp = lnp_counts.drop_duplicates('cell')[['cell', 'lnp_call']].rename(columns={'lnp_call': 'dominant_lnp_call'})

    out = counts.merge(summary, on='cell', how='left')
    out = out.merge(dominant_bc[['cell', 'dominant_barcode', 'dominant_barcode_count']], on='cell', how='left')
    out = out.merge(dominant_lnp, on='cell', how='left')
    return out


## 4. Load the fixed detection parameters

These are the parameters used for the manuscript analysis. Calibration and parameter-sweep sections from the exploratory notebook are intentionally omitted.


In [ ]:
def batch_from_region(region: str) -> str:
    return str(region).split('_', 1)[0]


DETECTION_BATCHES = sorted({batch_from_region(info['region']) for info in region_info})

# Fixed manual defaults. Use one uniform LoG threshold for every barcode channel;
# per-channel calibrated thresholds missed too many visually obvious dots.
DEFAULT_MANUAL_DETECTION_PARAMS = dict(
    threshold_peaks=1.8,
    min_on_snr=0.8,
)


# Optional batch-to-batch overrides. Apply manual LoG thresholds for the requested batches.
MANUAL_BATCH_OVERRIDES = {
    'S3': {
        'threshold_peaks_by_marker': {
            'A6': 1.6,
            'A17': 0.8,
            'A72': 1.3,
            'A63': 1.3,
            'A55': 2.0,
            'A76': 3.0,
            'A79': 3.0,
            'A20': 2.8,
        },
    },
    'S4': {
        'threshold_peaks_by_marker': {
            'A6': 1.6,
            'A17': 0.8,
            'A72': 1.3,
            'A63': 1.3,
            'A55': 2.0,
            'A76': 3.0,
            'A79': 3.0,
            'A20': 2.8,
        },
    },
}


def detection_batch_for_region(region: str) -> str:
    batch = batch_from_region(region)
    if batch not in DETECTION_BATCHES:
        raise KeyError(f'No manual detection batch registered for region {region}')
    return batch


def active_log_sigmas() -> Tuple[float, ...]:
    return tuple(P.log_sigmas) if P.log_sigmas else (P.log_sigma,)


def build_hot_mask(stack, region_markers: List[str], marker_to_idx: Dict[str, int], roi_yx=None):
    def crop(img):
        return img if roi_yx is None else img[roi_yx[0], roi_yx[1]]
    if not region_markers:
        return np.zeros(crop(stack[0]).shape, dtype=bool)
    hot_count = np.zeros(crop(stack[marker_to_idx[region_markers[0]]]).shape, dtype=np.uint8)
    for marker in region_markers:
        hot_count += crop(stack[marker_to_idx[marker]]) >= P.hotpx_sat_value
    return hot_count >= P.hotpx_count_threshold


def write_raw_max_for_markers(stack, region_markers: List[str], marker_to_idx: Dict[str, int],
                              out_path: Path, roi_yx=None, tissue_mask=None,
                              tile_size: int = 4096):
    """Write a memory-mapped channel maximum without materializing a full float stack."""
    if not region_markers:
        raise ValueError('Cannot build a raw maximum without barcode markers')
    first = apply_roi(stack[marker_to_idx[region_markers[0]]], roi_yx)
    if first.ndim != 2:
        raise ValueError(f'Expected 2D marker images, got shape {first.shape}')
    if tissue_mask is not None and tissue_mask.shape != first.shape:
        raise ValueError(
            f'Tissue-mask shape {tissue_mask.shape} does not match raw image shape {first.shape}'
        )
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    raw_max = tifffile.memmap(
        out_path, shape=first.shape, dtype=first.dtype, photometric='minisblack'
    )
    rows_per_tile = max(1, int(tile_size))
    for y0 in range(0, first.shape[0], rows_per_tile):
        y1 = min(first.shape[0], y0 + rows_per_tile)
        block = np.array(first[y0:y1], copy=True)
        for marker in region_markers[1:]:
            marker_img = apply_roi(stack[marker_to_idx[marker]], roi_yx)
            np.maximum(block, marker_img[y0:y1], out=block)
        if tissue_mask is not None:
            block[~np.asarray(tissue_mask[y0:y1], dtype=bool)] = 0
        raw_max[y0:y1] = block
    raw_max.flush()
    return raw_max


def log_filter_scale_normalized(img: np.ndarray, sigma: float) -> np.ndarray:
    sigma = float(sigma)
    if globals().get('USE_GPU_SPOT_DETECTION', False) and globals().get('cp') is not None and globals().get('cndi') is not None:
        try:
            img_gpu = cp.asarray(img, dtype=cp.float32)
            score_gpu = -cndi.gaussian_laplace(img_gpu, sigma=sigma) * sigma ** 2
            return cp.asnumpy(score_gpu)
        except Exception as exc:
            _disable_gpu_spot_detection(exc)
    return -ndi.gaussian_laplace(np.asarray(img, dtype=np.float32), sigma=sigma) * sigma ** 2


def manual_detection_params_for_batch(batch: str) -> dict:
    params = dict(DEFAULT_MANUAL_DETECTION_PARAMS)
    params.update(MANUAL_BATCH_OVERRIDES.get(batch, {}))
    by_marker = params.get('threshold_peaks_by_marker', {}) or {}
    per_marker = {}
    for marker in BARCODE_MARKERS:
        per_marker[marker] = dict(
            threshold_peaks=float(by_marker.get(marker, params['threshold_peaks'])),
            threshold_source='manual_batch_override' if marker in by_marker else 'manual_batch_default',
        )
    return dict(
        source='manual_uniform_thresholds_no_channel_calibration',
        batch=batch,
        log_sigma=P.log_sigma,
        log_sigmas=active_log_sigmas(),
        peak_width=P.peak_width,
        nms_min_distance=P.nms_min_distance,
        expected_on_bits=P.expected_on_bits,
        min_on_snr=float(params['min_on_snr']),
        min_snr_margin=P.min_snr_margin,
        raw_peak_percentile=P.raw_peak_percentile,
        raw_object_percentile=P.raw_object_percentile,
        barcode_max_hamming_distance=P.barcode_max_hamming_distance,
        bright_snr_threshold=P.bright_snr_threshold,
        max_bright_markers=P.max_bright_markers,
        per_marker_log_thresholds=per_marker,
    )


SELECTED_DETECTION_PARAMS = {
    batch: manual_detection_params_for_batch(batch)
    for batch in DETECTION_BATCHES
}


def selected_detection_params_for_region(region: str) -> dict:
    return SELECTED_DETECTION_PARAMS[detection_batch_for_region(region)]


def marker_thresholds_for_region(region: str) -> dict:
    return selected_detection_params_for_region(region)['per_marker_log_thresholds']


def min_on_snr_for_region(region: str) -> float:
    return float(selected_detection_params_for_region(region)['min_on_snr'])


def threshold_for_marker(marker: str, log_thresholds: Optional[dict] = None) -> float:
    active = log_thresholds if log_thresholds is not None else globals().get('PER_MARKER_LOG_THRESHOLDS', {})
    if marker in active:
        return float(active[marker].get('threshold_peaks', P.threshold_peaks))
    return float(P.threshold_peaks)


def sample_tissue_fields(info: dict, n_fields: int, field_size: int, rng, min_tissue_fraction: float = 0.10) -> List[Tuple[int, int]]:
    h = int(info['shape'][1])
    w = int(info['shape'][2])
    field_size = int(min(field_size, h, w))
    max_y0 = max(0, h - field_size)
    max_x0 = max(0, w - field_size)
    tissue_mask = np.load(info['tissue_mask'], mmap_mode='r').astype(bool) if info['tissue_mask'].exists() else None

    fields = []
    tried = set()
    max_attempts = max(200, n_fields * 80)
    for _ in range(max_attempts):
        if len(fields) >= n_fields:
            break
        y0 = int(rng.integers(0, max_y0 + 1)) if max_y0 else 0
        x0 = int(rng.integers(0, max_x0 + 1)) if max_x0 else 0
        key = (y0, x0)
        if key in tried:
            continue
        tried.add(key)
        if tissue_mask is not None:
            crop_mask = tissue_mask[y0:y0 + field_size, x0:x0 + field_size]
            if crop_mask.size and float(np.mean(crop_mask)) < min_tissue_fraction:
                continue
        fields.append(key)

    if len(fields) < n_fields:
        log.warning(f"{info['region']}: sampled only {len(fields)}/{n_fields} QC fields")
    return fields


manual_detection_params_v4 = pd.DataFrame([
    dict(
        batch=batch,
        threshold_peaks=SELECTED_DETECTION_PARAMS[batch]['per_marker_log_thresholds'][BARCODE_MARKERS[0]]['threshold_peaks'],
        min_on_snr=SELECTED_DETECTION_PARAMS[batch]['min_on_snr'],
        min_snr_margin=SELECTED_DETECTION_PARAMS[batch]['min_snr_margin'],
        raw_peak_percentile=SELECTED_DETECTION_PARAMS[batch]['raw_peak_percentile'],
        raw_object_percentile=SELECTED_DETECTION_PARAMS[batch]['raw_object_percentile'],
        barcode_max_hamming_distance=SELECTED_DETECTION_PARAMS[batch]['barcode_max_hamming_distance'],
        log_sigma=SELECTED_DETECTION_PARAMS[batch]['log_sigma'],
        log_sigmas=','.join(map(str, SELECTED_DETECTION_PARAMS[batch].get('log_sigmas', [SELECTED_DETECTION_PARAMS[batch]['log_sigma']]))),
        peak_width=SELECTED_DETECTION_PARAMS[batch]['peak_width'],
        nms_min_distance=SELECTED_DETECTION_PARAMS[batch]['nms_min_distance'],
    )
    for batch in DETECTION_BATCHES
])

(OUTDIR / 'selected_detection_parameters_manual_v4.json').write_text(json.dumps(SELECTED_DETECTION_PARAMS, indent=2))
# Keep the legacy filename available for downstream helper cells or separately-run notebook sections.
(OUTDIR / 'selected_detection_parameters_v4.json').write_text(json.dumps(SELECTED_DETECTION_PARAMS, indent=2))
manual_detection_params_v4
display(manual_detection_params_v4)


## 5. Define full-barcode candidate detection and decoding


In [ ]:
# Shared candidate-detection helpers used by calibration, random-field QC, and full-region processing.
def apply_roi(img2d: np.ndarray, roi_yx=None):
    return img2d if roi_yx is None else img2d[roi_yx[0], roi_yx[1]]


def greedy_nms(coords_ij: np.ndarray, scores: np.ndarray, min_distance: float) -> np.ndarray:
    if len(coords_ij) == 0:
        return np.empty((0, 2), dtype=np.int64)
    order = np.argsort(scores)[::-1]
    keep = []
    min_d2 = float(min_distance) ** 2
    for pt in np.asarray(coords_ij)[order]:
        if not keep or np.all(np.sum((np.asarray(keep) - pt) ** 2, axis=1) >= min_d2):
            keep.append(pt)
    return np.asarray(keep, dtype=np.int64).reshape(-1, 2)


def build_hot_mask(stack, region_markers: List[str], marker_to_idx: Dict[str, int], roi_yx=None):
    if not region_markers:
        shape = apply_roi(stack[0], roi_yx).shape
        return np.zeros(shape, dtype=bool)
    shape = apply_roi(stack[marker_to_idx[region_markers[0]]], roi_yx).shape
    hot_count = np.zeros(shape, dtype=np.uint8)
    for marker in region_markers:
        hot_count += apply_roi(stack[marker_to_idx[marker]], roi_yx) >= P.hotpx_sat_value
    return hot_count >= P.hotpx_count_threshold


def log_filter_scale_normalized(img: np.ndarray, sigma: float) -> np.ndarray:
    # Multiplication by sigma**2 makes LoG responses comparable across spot sizes.
    sigma = float(sigma)
    if globals().get('USE_GPU_SPOT_DETECTION', False) and globals().get('cp') is not None and globals().get('cndi') is not None:
        try:
            img_gpu = cp.asarray(img, dtype=cp.float32)
            score_gpu = -cndi.gaussian_laplace(img_gpu, sigma=sigma) * sigma ** 2
            return cp.asnumpy(score_gpu)
        except Exception as exc:
            _disable_gpu_spot_detection(exc)
    return -ndi.gaussian_laplace(np.asarray(img, dtype=np.float32), sigma=sigma) * sigma ** 2


def detect_marker_log_candidates(img: np.ndarray, marker: str, threshold: float,
                                 hot_mask=None, tissue_mask=None) -> pd.DataFrame:
    tables = []
    for sigma in active_log_sigmas():
        score = log_filter_scale_normalized(img, sigma)
        valid = np.isfinite(score)
        if hot_mask is not None and hot_mask.shape == score.shape:
            valid &= ~hot_mask
        if tissue_mask is not None and tissue_mask.shape == score.shape:
            valid &= tissue_mask
        peaks = valid & (score >= float(threshold))
        peaks &= score == ndi.maximum_filter(score, size=P.peak_width, mode='nearest')
        coords = np.argwhere(peaks)
        if len(coords):
            tables.append(pd.DataFrame({'i': coords[:, 0], 'j': coords[:, 1],
                                        'score': score[peaks], 'seed_marker': marker,
                                        'candidate_source': 'log', 'log_sigma': float(sigma)}))
    if not tables:
        return pd.DataFrame(columns=['i', 'j', 'score', 'seed_marker', 'candidate_source', 'log_sigma'])
    found = pd.concat(tables, ignore_index=True)
    kept = greedy_nms(found[['i', 'j']].to_numpy(), found['score'].to_numpy(), P.nms_min_distance)
    keys = pd.DataFrame(kept, columns=['i', 'j']).assign(_keep=True)
    return (found.merge(keys, on=['i', 'j']).sort_values('score', ascending=False)
                 .drop_duplicates(['i', 'j']).drop(columns='_keep').reset_index(drop=True))


def raw_rescue_candidate_tables(raw_max: np.ndarray, hot_mask=None, out_dir=None, prefix='raw'):
    img = np.asarray(raw_max)
    valid = np.isfinite(img) & (img > 0)
    if hot_mask is not None and hot_mask.shape == img.shape:
        valid &= ~hot_mask
    tables = []
    counts = {'raw_bright_threshold': np.nan, 'n_raw_bright_candidates': 0,
              'raw_object_threshold': np.nan, 'n_raw_object_candidates': 0}
    vals = img[valid]
    if not vals.size:
        return tables, counts
    if P.use_raw_bright_rescue:
        thr = float(np.percentile(vals, P.raw_peak_percentile))
        mask = valid & (img >= thr) & (img == ndi.maximum_filter(img, size=P.peak_width, mode='nearest'))
        coords = np.argwhere(mask)
        scores = img[mask].astype(np.float32)
        kept = greedy_nms(coords, scores, P.nms_min_distance)
        if len(kept):
            tbl = pd.DataFrame({'i': kept[:, 0], 'j': kept[:, 1], 'score': img[kept[:, 0], kept[:, 1]],
                                'seed_marker': 'raw_bright', 'candidate_source': 'raw_bright', 'log_sigma': np.nan})
            tables.append(tbl)
        counts.update(raw_bright_threshold=thr, n_raw_bright_candidates=len(kept))
    if P.use_raw_object_rescue and not P.skip_raw_object_rescue:
        thr = float(np.percentile(vals, P.raw_object_percentile))
        labels, nlab = ndi.label(valid & (img >= thr))
        objects = ndi.find_objects(labels)
        rows = []
        for label_id, slc in enumerate(objects, 1):
            if slc is None:
                continue
            area = int(np.count_nonzero(labels[slc] == label_id))
            if P.raw_object_min_area <= area <= P.raw_object_max_area:
                yy, xx = np.where(labels[slc] == label_id)
                i = int(round(float(yy.mean() + slc[0].start)))
                j = int(round(float(xx.mean() + slc[1].start)))
                rows.append((i, j, float(img[i, j])))
        if rows:
            tbl = pd.DataFrame(rows, columns=['i', 'j', 'score'])
            tbl['seed_marker'] = 'raw_object'; tbl['candidate_source'] = 'raw_object'; tbl['log_sigma'] = np.nan
            tables.append(tbl)
        counts.update(raw_object_threshold=thr, n_raw_object_candidates=len(rows))
    return tables, counts


def merge_marker_candidates(candidate_tables: List[pd.DataFrame]) -> pd.DataFrame:
    nonempty = [x for x in candidate_tables if x is not None and len(x)]
    if not nonempty:
        return pd.DataFrame(columns=['i', 'j', 'score', 'seed_marker', 'candidate_source'])
    merged = pd.concat(nonempty, ignore_index=True)
    kept = greedy_nms(merged[['i', 'j']].to_numpy(), merged['score'].to_numpy(), P.nms_min_distance)
    keys = pd.DataFrame(kept, columns=['i', 'j']).assign(_keep=True)
    return (merged.merge(keys, on=['i', 'j']).sort_values('score', ascending=False)
                  .drop_duplicates(['i', 'j']).drop(columns='_keep').reset_index(drop=True))


def _snr_offsets():
    r = int(P.snr_annulus_outer)
    yy, xx = np.meshgrid(np.arange(-r, r + 1), np.arange(-r, r + 1), indexing='ij')
    rr = np.maximum(np.abs(yy), np.abs(xx))
    return yy, xx, rr <= P.snr_core_radius, (rr >= P.snr_annulus_inner) & (rr <= P.snr_annulus_outer)


def decode_spots_full_barcode(stack, region_markers: List[str], marker_to_idx: Dict[str, int],
                          spots: pd.DataFrame, roi_yx=None, min_on_snr: Optional[float] = None) -> pd.DataFrame:
    spots = spots.reset_index(drop=True).copy()
    empty_cols = ['called_code', 'decoded_on_markers', 'min_on_snr', 'max_off_snr', 'snr_margin',
                  'n_bright_markers', 'promiscuous', 'matched_barcode', 'lnp_call',
                  'barcode_match_distance', 'barcode_match_status', 'barcode_in_library',
                  'barcode_excluded', 'pass_quality', 'accepted', 'accept_reason']
    if spots.empty:
        for col in empty_cols: spots[col] = pd.Series(dtype=object)
        return spots
    dy, dx, core, ann = _snr_offsets()
    ii = spots['i'].to_numpy(int); jj = spots['j'].to_numpy(int)
    snr = np.zeros((len(spots), len(region_markers)), dtype=np.float32)
    for mi, marker in enumerate(region_markers):
        img = apply_roi(stack[marker_to_idx[marker]], roi_yx)
        yy = np.clip(ii[:, None, None] + dy, 0, img.shape[0] - 1)
        xx = np.clip(jj[:, None, None] + dx, 0, img.shape[1] - 1)
        blocks = np.asarray(img[yy, xx], dtype=np.float32)
        signal = blocks[:, core].max(axis=1); background = np.median(blocks[:, ann], axis=1)
        snr[:, mi] = (signal + 1.0) / (background + 1.0)
        spots[f'raw_signal_{marker}'] = signal; spots[f'raw_background_{marker}'] = background
        spots[f'local_snr_{marker}'] = snr[:, mi]
    k = min(int(P.expected_on_bits), len(region_markers))
    top = np.argpartition(snr, -k, axis=1)[:, -k:]
    called = np.zeros_like(snr, dtype=np.uint8)
    np.put_along_axis(called, top, 1, axis=1)
    codes = [''.join(map(str, row)) for row in called]
    sorted_snr = np.sort(snr, axis=1)
    on_snr = sorted_snr[:, -k]; off_snr = sorted_snr[:, -(k + 1)] if len(region_markers) > k else np.zeros(len(spots))
    decode_rows = []
    for code in codes:
        distances = [(bc, name, sum(a != b for a, b in zip(code, bc))) for bc, name in BARCODE_LIBRARY.items()]
        dmin = min(x[2] for x in distances); hits = [x for x in distances if x[2] == dmin and dmin <= P.barcode_max_hamming_distance]
        if len(hits) == 1:
            bc, name, dist = hits[0]; status = 'exact' if dist == 0 else 'tolerant'
            decode_rows.append((bc, name, dist, status, dist == 0, False))
        elif len(hits) > 1:
            decode_rows.append(('', 'ambiguous_mixed', dmin, 'ambiguous_mixed', False, True))
        else:
            decode_rows.append(('', 'unmapped', dmin, 'unmapped', False, False))
    decoded = pd.DataFrame(decode_rows, columns=['matched_barcode', 'lnp_call', 'barcode_match_distance',
                                                    'barcode_match_status', 'barcode_in_library', 'barcode_excluded'])
    spots['called_code'] = codes
    spots['decoded_on_markers'] = [', '.join(m for m, bit in zip(region_markers, code) if bit == '1') for code in codes]
    spots['min_on_snr'] = on_snr; spots['max_off_snr'] = off_snr; spots['snr_margin'] = on_snr - off_snr
    spots['n_bright_markers'] = (snr > P.bright_snr_threshold).sum(axis=1)
    spots['promiscuous'] = spots['n_bright_markers'] > P.max_bright_markers
    for col in decoded: spots[col] = decoded[col]
    active_min_snr = float(P.min_on_snr if min_on_snr is None else min_on_snr)
    spots['pass_quality'] = spots['min_on_snr'] >= active_min_snr
    spots['accepted'] = (spots['pass_quality'] & ~spots['promiscuous'] &
                         (spots['snr_margin'] >= P.min_snr_margin) &
                         spots['barcode_match_status'].isin(['exact', 'tolerant']))
    spots['accept_reason'] = np.where(spots['accepted'], 'accepted',
        np.where(~spots['barcode_match_status'].isin(['exact', 'tolerant']), 'decode_' + spots['barcode_match_status'],
        np.where(spots['promiscuous'], 'promiscuous_multi_marker',
        np.where(~spots['pass_quality'], 'below_min_on_snr', 'low_snr_margin'))))
    return spots


# Keep raw-rescue percentile settings valid in already-running kernels.
if not (0 <= float(P.raw_peak_percentile) <= 100):
    if float(P.raw_peak_percentile) == 980:
        log.warning('Correcting P.raw_peak_percentile from 980 to 98.0')
        P.raw_peak_percentile = 98.0
    else:
        raise ValueError(f'P.raw_peak_percentile must be in [0, 100], got {P.raw_peak_percentile}')
if not (0 <= float(P.raw_object_percentile) <= 100):
    raise ValueError(f'P.raw_object_percentile must be in [0, 100], got {P.raw_object_percentile}')



def write_raw_max_for_markers(stack, region_markers: List[str], marker_to_idx: Dict[str, int],
                              out_path: Path, roi_yx=None, tissue_mask=None,
                              tile_size: int = 4096):
    """Write a memory-mapped channel maximum without materializing a full float stack."""
    if not region_markers:
        raise ValueError('Cannot build a raw maximum without barcode markers')
    first = apply_roi(stack[marker_to_idx[region_markers[0]]], roi_yx)
    if first.ndim != 2:
        raise ValueError(f'Expected 2D marker images, got shape {first.shape}')
    if tissue_mask is not None and tissue_mask.shape != first.shape:
        raise ValueError(
            f'Tissue-mask shape {tissue_mask.shape} does not match raw image shape {first.shape}'
        )
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    raw_max = tifffile.memmap(
        out_path, shape=first.shape, dtype=first.dtype, photometric='minisblack'
    )
    rows_per_tile = max(1, int(tile_size))
    for y0 in range(0, first.shape[0], rows_per_tile):
        y1 = min(first.shape[0], y0 + rows_per_tile)
        block = np.array(first[y0:y1], copy=True)
        for marker in region_markers[1:]:
            marker_img = apply_roi(stack[marker_to_idx[marker]], roi_yx)
            np.maximum(block, marker_img[y0:y1], out=block)
        if tissue_mask is not None:
            block[~np.asarray(tissue_mask[y0:y1], dtype=bool)] = 0
        raw_max[y0:y1] = block
    raw_max.flush()
    return raw_max


def robust_limits(img: np.ndarray, p_low=1, p_high=99.8):
    finite = np.asarray(img)[np.isfinite(img)]
    if finite.size == 0:
        return 0.0, 1.0
    return tuple(np.percentile(finite, [p_low, p_high]))


## 6. Configure optional GPU acceleration

No package installation is performed in the notebook. If CuPy is unavailable, the same operations use the CPU implementation.


In [ ]:
USE_GPU_SPOT_DETECTION = True
GPU_SPOT_BACKEND = 'cpu'
GPU_SPOT_FALLBACK_REASON = ''
_GPU_SPOT_WARNED = False

try:
    import cupy as cp
    from cupyx.scipy import ndimage as cndi

    n_gpu = int(cp.cuda.runtime.getDeviceCount())
    if n_gpu < 1:
        raise RuntimeError('CuPy imported, but no CUDA GPU was detected')
    _ = cp.zeros((1,), dtype=cp.float32)
    GPU_SPOT_BACKEND = 'cupy'
    log.info(f'GPU spot detection enabled with CuPy; CUDA devices detected: {n_gpu}')
except Exception as exc:
    cp = None
    cndi = None
    USE_GPU_SPOT_DETECTION = False
    GPU_SPOT_FALLBACK_REASON = repr(exc)
    log.warning(f'GPU spot detection unavailable; using CPU scipy.ndimage. Reason: {GPU_SPOT_FALLBACK_REASON}')


def _disable_gpu_spot_detection(exc: Exception) -> None:
    global USE_GPU_SPOT_DETECTION, GPU_SPOT_BACKEND, GPU_SPOT_FALLBACK_REASON, _GPU_SPOT_WARNED
    USE_GPU_SPOT_DETECTION = False
    GPU_SPOT_BACKEND = 'cpu'
    GPU_SPOT_FALLBACK_REASON = repr(exc)
    if not _GPU_SPOT_WARNED:
        log.warning(f'GPU spot detection failed during execution; falling back to CPU. Reason: {GPU_SPOT_FALLBACK_REASON}')
        _GPU_SPOT_WARNED = True


def log_filter(img: np.ndarray, sigma: Optional[float] = None) -> np.ndarray:
    sigma = float(P.log_sigma if sigma is None else sigma)
    if USE_GPU_SPOT_DETECTION and cp is not None and cndi is not None:
        try:
            img_gpu = cp.asarray(img, dtype=cp.float32)
            score_gpu = -cndi.gaussian_laplace(img_gpu, sigma=float(sigma))
            return cp.asnumpy(score_gpu)
        except Exception as exc:
            _disable_gpu_spot_detection(exc)
    return -ndi.gaussian_laplace(img.astype(np.float32, copy=False), sigma=sigma)


def find_spots_from_score(score: np.ndarray, threshold: float) -> pd.DataFrame:
    if USE_GPU_SPOT_DETECTION and cp is not None and cndi is not None:
        try:
            score_gpu = cp.asarray(score, dtype=cp.float32)
            local_max_gpu = score_gpu == cndi.maximum_filter(score_gpu, size=P.peak_width, mode='nearest')
            above_gpu = score_gpu >= float(threshold)
            coords_gpu = cp.argwhere(local_max_gpu & above_gpu)
            coords = cp.asnumpy(coords_gpu).astype(np.int64, copy=False)
            if len(coords):
                raw_scores = cp.asnumpy(score_gpu[coords_gpu[:, 0], coords_gpu[:, 1]]).astype(np.float32, copy=False)
            else:
                raw_scores = np.array([], dtype=np.float32)
            kept = greedy_nms(coords, raw_scores, P.nms_min_distance)
            kept_scores = score[kept[:, 0], kept[:, 1]] if len(kept) else np.array([], dtype=np.float32)
            return pd.DataFrame({
                'i': kept[:, 0] if len(kept) else [],
                'j': kept[:, 1] if len(kept) else [],
                'score': kept_scores,
            })
        except Exception as exc:
            _disable_gpu_spot_detection(exc)

    local_max = score == ndi.maximum_filter(score, size=P.peak_width, mode='nearest')
    above = score >= threshold
    coords = np.argwhere(local_max & above)
    raw_scores = score[coords[:, 0], coords[:, 1]] if len(coords) else np.array([], dtype=np.float32)
    kept = greedy_nms(coords, raw_scores, P.nms_min_distance)
    kept_scores = score[kept[:, 0], kept[:, 1]] if len(kept) else np.array([], dtype=np.float32)
    return pd.DataFrame({
        'i': kept[:, 0] if len(kept) else [],
        'j': kept[:, 1] if len(kept) else [],
        'score': kept_scores,
    })


print(f'Section 6 spot-detection backend: {GPU_SPOT_BACKEND}')
if GPU_SPOT_FALLBACK_REASON:
    print(f'GPU fallback reason: {GPU_SPOT_FALLBACK_REASON}')


## 7. Detect and decode spots for all retained regions

This is the long-running step. It processes each region in overlapping tiles and writes checkpoint and summary CSV files. The notebook has been delivered without executing this step because the TIFF stacks are hundreds of gigabytes.


In [ ]:
RUN_REGIONS = None          # Example: ['S1_reg000', 'S1_reg001']; None runs every region.
FORCE_RERUN_REGION = False  # Set True to overwrite existing per-region checkpoint files.
SKIP_COMPLETED_REGIONS = True
CLEAR_GPU_MEMORY = True     # Also clear CuPy's memory pool between regions when GPU detection is active.
REGION_N_TILES = 10         # Split each region into this many y-axis tiles for safer raw-object rescue.
REGION_TILE_OVERLAP = 128   # Overlap protects spots near tile boundaries; only tile cores are kept.
WRITE_TILE_CHECKPOINTS = True


def region_checkpoint_paths(region: str) -> Dict[str, Path]:
    return {
        'all_candidates': OUTDIR / f'spots_all_candidates_{region}_v4.csv',
        'accepted': OUTDIR / f'spots_{region}_v4.csv',
        'cell_counts': OUTDIR / f'cell_spot_counts_{region}_v4.csv',
        'image_qa': OUTDIR / f'qa_per_image_{region}_v4.csv',
        'spot_qa': OUTDIR / f'qa_spot_detection_{region}_v4.csv',
    }


REQUIRED_REGION_CHECKPOINT_KEYS = ('all_candidates', 'accepted', 'cell_counts')


def region_checkpoint_status(region: str) -> Tuple[bool, Dict[str, Path]]:
    paths = region_checkpoint_paths(region)
    missing_required = {key: paths[key] for key in REQUIRED_REGION_CHECKPOINT_KEYS if not paths[key].exists()}
    return len(missing_required) == 0, missing_required


def region_checkpoint_complete(region: str) -> bool:
    complete, _ = region_checkpoint_status(region)
    return complete


def _current_rss_gb() -> Optional[float]:
    try:
        with open('/proc/self/status') as f:
            for line in f:
                if line.startswith('VmRSS:'):
                    return int(line.split()[1]) / (1024 ** 2)
    except Exception:
        return None
    return None


def clear_region_memory(*objects):
    for obj in objects:
        try:
            del obj
        except Exception:
            pass
    gc.collect()
    try:
        import ctypes
        ctypes.CDLL('libc.so.6').malloc_trim(0)  # glibc keeps freed heap pages unless told to return them
    except Exception as exc:
        log.debug(f'Could not malloc_trim: {exc}')
    if CLEAR_GPU_MEMORY and 'cp' in globals() and cp is not None:
        try:
            cp.get_default_memory_pool().free_all_blocks()
            cp.get_default_pinned_memory_pool().free_all_blocks()
        except Exception as exc:
            log.debug(f'Could not clear GPU memory pool: {exc}')


def apply_roi(img2d: np.ndarray, roi_yx=None):
    """Return a 2D image unchanged or cropped to a (y, x) slice pair."""
    return img2d if roi_yx is None else img2d[roi_yx[0], roi_yx[1]]


def image_qa(img: np.ndarray, label: str) -> dict:
    vals = np.asarray(img)
    finite = vals[np.isfinite(vals)]
    if finite.size == 0:
        raise ValueError(f'Cannot compute image QA for {label}: image has no finite pixels')
    med = float(np.median(finite))
    mad = float(np.median(np.abs(finite - med)))
    p = np.percentile(finite, [1, 99, 99.9, 99.99])
    sat = float((finite >= P.hotpx_sat_value).mean())
    dr = float(p[3] / max(med, 1.0))
    snr = float((p[2] - med) / max(1.4826 * mad, 1.0))
    fg = float((finite > 5 * max(med, 1.0)).mean())
    return dict(
        label=label, shape=str(vals.shape), dtype=str(vals.dtype), min=float(finite.min()),
        median=med, mean=float(finite.mean()), max=float(finite.max()), p1=float(p[0]),
        p99=float(p[1]), p999=float(p[2]), p9999=float(p[3]), sat_frac=sat,
        dynamic_range=dr, foreground_frac=fg, snr_proxy=snr,
    )


def load_cell_features(path: Path, roi_yx=None) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {'label', 'y', 'x'}
    if not required.issubset(df.columns):
        raise ValueError(f'{path} must contain columns {required}; found {df.columns[:20].tolist()}')
    if roi_yx is not None:
        ys, xs = roi_yx
        y0 = 0 if ys.start is None else ys.start
        y1 = np.inf if ys.stop is None else ys.stop
        x0 = 0 if xs.start is None else xs.start
        x1 = np.inf if xs.stop is None else xs.stop
        df = df[(df['y'] >= y0) & (df['y'] < y1) & (df['x'] >= x0) & (df['x'] < x1)].copy()
        df['y'] -= y0
        df['x'] -= x0
    return df


def assign_spots_to_nearest_cell(spots: pd.DataFrame, cells: pd.DataFrame) -> pd.DataFrame:
    out = spots.copy()
    out['cell'] = 0
    out['cell_distance_px'] = np.nan
    if len(out) == 0 or len(cells) == 0:
        return out
    tree = cKDTree(cells[['y', 'x']].to_numpy(float))
    dist, idx = tree.query(
        out[['i', 'j']].to_numpy(float), distance_upper_bound=P.cell_assignment_radius_px
    )
    ok = np.isfinite(dist) & (idx < len(cells))
    labels = cells['label'].to_numpy()
    out.loc[ok, 'cell'] = labels[idx[ok]]
    out.loc[ok, 'cell_distance_px'] = dist[ok]
    return out



def _slice_bounds(s: slice, max_len: int) -> Tuple[int, int]:
    start = 0 if s.start is None else int(s.start)
    stop = int(max_len) if s.stop is None else int(s.stop)
    return max(0, start), min(int(max_len), stop)


def make_region_processing_tiles(stack_shape, roi_yx=None, n_tiles: int = 10, overlap: int = 128) -> List[dict]:
    if roi_yx is None:
        y_start, y_stop = 0, int(stack_shape[1])
        x_start, x_stop = 0, int(stack_shape[2])
    else:
        y_start, y_stop = _slice_bounds(roi_yx[0], int(stack_shape[1]))
        x_start, x_stop = _slice_bounds(roi_yx[1], int(stack_shape[2]))
    height = max(0, y_stop - y_start)
    if height == 0 or x_stop <= x_start:
        return []
    n_tiles = max(1, min(int(n_tiles), height))
    edges = np.linspace(y_start, y_stop, n_tiles + 1).round().astype(int)
    tiles = []
    for tile_idx in range(n_tiles):
        core_y0 = int(edges[tile_idx])
        core_y1 = int(edges[tile_idx + 1])
        if core_y1 <= core_y0:
            continue
        tile_y0 = max(y_start, core_y0 - int(overlap))
        tile_y1 = min(y_stop, core_y1 + int(overlap))
        tiles.append(dict(
            tile_idx=tile_idx,
            roi_yx=(slice(tile_y0, tile_y1), slice(x_start, x_stop)),
            core_y0=core_y0,
            core_y1=core_y1,
            x0=x_start,
            x1=x_stop,
        ))
    return tiles


def tile_checkpoint_paths(region: str, tile_idx: int) -> Dict[str, Path]:
    tile_dir = OUTDIR / 'region_tile_checkpoints' / region
    return {
        'all_candidates': tile_dir / f'spots_all_candidates_{region}_tile{tile_idx:03d}_v4.csv',
        'accepted': tile_dir / f'spots_{region}_tile{tile_idx:03d}_v4.csv',
        'raw_qa': tile_dir / f'raw_rescue_qa_{region}_tile{tile_idx:03d}_v4.csv',
    }


def _read_tile_if_complete(region: str, tile_idx: int):
    paths = tile_checkpoint_paths(region, tile_idx)
    if not (paths['all_candidates'].exists() and paths['accepted'].exists() and paths['raw_qa'].exists()):
        return None
    spots = pd.read_csv(paths['all_candidates'])
    accepted = pd.read_csv(paths['accepted'])
    raw_qa = pd.read_csv(paths['raw_qa']).iloc[0].to_dict()
    return spots, accepted, raw_qa


def _offset_tile_spots_to_global(spots: pd.DataFrame, tile: dict) -> pd.DataFrame:
    if spots is None or len(spots) == 0:
        return spots
    out = spots.copy()
    tile_y0 = int(tile['roi_yx'][0].start)
    tile_x0 = int(tile['roi_yx'][1].start)
    out['i_tile'] = out['i'].astype(int)
    out['j_tile'] = out['j'].astype(int)
    out['i_global'] = out['i_tile'] + tile_y0
    out['j_global'] = out['j_tile'] + tile_x0
    keep = (
        (out['i_global'] >= int(tile['core_y0']))
        & (out['i_global'] < int(tile['core_y1']))
        & (out['j_global'] >= int(tile['x0']))
        & (out['j_global'] < int(tile['x1']))
    )
    out = out.loc[keep].copy()
    # Downstream cell assignment expects full-region coordinates in i/j.
    out['i'] = out['i_global'].astype(int)
    out['j'] = out['j_global'].astype(int)
    return out.reset_index(drop=True)


def process_region_tile_spot_detection_v4(
    r: dict,
    stack,
    markers: List[str],
    marker_to_idx: Dict[str, int],
    region_markers: List[str],
    tissue_mask_full,
    region_log_thresholds: dict,
    region_min_on_snr: float,
    tile: dict,
) -> Tuple[pd.DataFrame, pd.DataFrame, dict]:
    region = r['region']
    batch = detection_batch_for_region(region)
    tile_idx = int(tile['tile_idx'])
    tile_paths = tile_checkpoint_paths(region, tile_idx)
    if WRITE_TILE_CHECKPOINTS:
        tile_paths['all_candidates'].parent.mkdir(parents=True, exist_ok=True)
        if not FORCE_RERUN_REGION:
            cached = _read_tile_if_complete(region, tile_idx)
            if cached is not None:
                log.info(f'{region} tile {tile_idx + 1}/{REGION_N_TILES}: using existing tile checkpoints')
                return cached

    tile_roi = tile['roi_yx']
    tile_label = f'tile{tile_idx:03d}_y{tile_roi[0].start}-{tile_roi[0].stop}'
    log.info(f'{region} {tile_label}: processing core y={tile["core_y0"]}:{tile["core_y1"]}')
    tile_hot_mask = build_hot_mask(stack, region_markers, marker_to_idx, tile_roi)
    tile_tissue_mask = None
    if tissue_mask_full is not None:
        tile_tissue_mask = np.asarray(tissue_mask_full[tile_roi[0], tile_roi[1]], dtype=bool)

    candidate_tables = []
    channel_candidate_counts = {}
    raw_rescue_counts = {
        'raw_bright_threshold': np.nan,
        'n_raw_bright_candidates': 0,
        'raw_object_threshold': np.nan,
        'n_raw_object_candidates': 0,
    }
    spots = pd.DataFrame()
    accepted = pd.DataFrame()
    try:
        for marker in region_markers:
            img = stack[marker_to_idx[marker], tile_roi[0], tile_roi[1]]
            thr = threshold_for_marker(marker, region_log_thresholds)
            marker_spots = detect_marker_log_candidates(
                img, marker, thr, hot_mask=tile_hot_mask, tissue_mask=tile_tissue_mask
            )
            channel_candidate_counts[marker] = len(marker_spots)
            if len(marker_spots):
                candidate_tables.append(marker_spots)
            del img, marker_spots
            clear_region_memory()

        if P.use_raw_bright_rescue or P.use_raw_object_rescue:
            raw_rescue_dir = OUTDIR / 'raw_rescue_candidates' / region
            raw_rescue_dir.mkdir(parents=True, exist_ok=True)
            raw_max_path = raw_rescue_dir / f'{region}_{tile_label}_raw_max_v4.tif'
            raw_max_mm = write_raw_max_for_markers(
                stack, region_markers, marker_to_idx, raw_max_path,
                roi_yx=tile_roi, tissue_mask=tile_tissue_mask, tile_size=P.raw_rescue_tile
            )
            raw_tables, raw_rescue_counts = raw_rescue_candidate_tables(
                raw_max_mm, hot_mask=tile_hot_mask, out_dir=raw_rescue_dir, prefix=f'{region}_{tile_label}_v4'
            )
            candidate_tables.extend(raw_tables)
            del raw_max_mm, raw_tables
            clear_region_memory()

        merged = merge_marker_candidates(candidate_tables)
        clear_region_memory(candidate_tables)
        candidate_tables = []

        spots = decode_spots_full_barcode(
            stack, region_markers, marker_to_idx, merged, roi_yx=tile_roi,
            min_on_snr=region_min_on_snr
        )
        del merged
        clear_region_memory()

        spots = _offset_tile_spots_to_global(spots, tile)
        if len(spots):
            spots['batch'] = batch
            spots['region'] = region
            spots['tile_idx'] = tile_idx
            spots['tile_core_y0'] = int(tile['core_y0'])
            spots['tile_core_y1'] = int(tile['core_y1'])
        accepted = spots[spots['accepted']].copy() if len(spots) else spots.copy()

        raw_qa = dict(
            region=region,
            batch=batch,
            tile_idx=tile_idx,
            tile_core_y0=int(tile['core_y0']),
            tile_core_y1=int(tile['core_y1']),
            n_hot_pixels=int(tile_hot_mask.sum()),
            n_channel_candidates_total=int(sum(channel_candidate_counts.values())),
            n_raw_bright_candidates=int(raw_rescue_counts.get('n_raw_bright_candidates', 0)),
            raw_bright_threshold=float(raw_rescue_counts.get('raw_bright_threshold', np.nan)),
            n_raw_object_candidates=int(raw_rescue_counts.get('n_raw_object_candidates', 0)),
            raw_object_threshold=float(raw_rescue_counts.get('raw_object_threshold', np.nan)),
            n_candidates=len(spots),
            n_spots=len(accepted),
        )

        if WRITE_TILE_CHECKPOINTS:
            spots.to_csv(tile_paths['all_candidates'], index=False)
            accepted.to_csv(tile_paths['accepted'], index=False)
            pd.DataFrame([raw_qa]).to_csv(tile_paths['raw_qa'], index=False)
        return spots, accepted, raw_qa
    finally:
        clear_region_memory(tile_hot_mask, tile_tissue_mask, candidate_tables, spots, accepted)


def process_region_spot_detection_v4(r: dict) -> None:
    region = r['region']
    batch = detection_batch_for_region(region)
    selected_params = selected_detection_params_for_region(region)
    region_log_thresholds = selected_params['per_marker_log_thresholds']
    region_min_on_snr = float(selected_params['min_on_snr'])
    sample = r['sample']
    paths = region_checkpoint_paths(region)

    log.info(f'===== {region} / {sample} / batch {batch} =====')
    t_region = tic()

    stack = None
    tissue_mask_full = None
    all_tile_spots = []
    all_tile_accepted = []
    tile_qa_rows = []
    spots = pd.DataFrame()
    accepted = pd.DataFrame()
    counts = pd.DataFrame()
    try:
        markers = r['markers']
        marker_to_idx = {m: i for i, m in enumerate(markers)}
        region_markers = [m for m in BARCODE_MARKERS if m in marker_to_idx]
        stack = open_stack(r['image'])
        log.info(f'image shape: {stack.shape}; barcode markers: {region_markers}')
        log.info(f'manual min_on_snr={region_min_on_snr:.3f}')

        if r['tissue_mask'].exists():
            tissue_mask_full = np.load(r['tissue_mask'], mmap_mode='r')

        qa_rows = []
        for marker in region_markers:
            img = apply_roi(stack[marker_to_idx[marker]], P.roi_yx)
            qa_rows.append({'region': region, 'batch': batch, 'sample': sample, 'marker': marker, **image_qa(img, marker)})
            del img
            clear_region_memory()
        pd.DataFrame(qa_rows).to_csv(paths['image_qa'], index=False)
        clear_region_memory(qa_rows)

        tiles = make_region_processing_tiles(
            stack.shape, roi_yx=P.roi_yx, n_tiles=REGION_N_TILES, overlap=REGION_TILE_OVERLAP
        )
        log.info(f'{region}: processing {len(tiles)} tiles with overlap={REGION_TILE_OVERLAP}')

        for tile in tiles:
            tile_spots, tile_accepted, raw_qa = process_region_tile_spot_detection_v4(
                r, stack, markers, marker_to_idx, region_markers, tissue_mask_full,
                region_log_thresholds, region_min_on_snr, tile
            )
            if len(tile_spots):
                all_tile_spots.append(tile_spots)
            if len(tile_accepted):
                all_tile_accepted.append(tile_accepted)
            tile_qa_rows.append(raw_qa)
            clear_region_memory(tile_spots, tile_accepted)
            rss_gb = _current_rss_gb()
            if rss_gb is not None:
                log.info(f'{region} tile {int(tile["tile_idx"]) + 1}: resident memory after cleanup = {rss_gb:.2f} GB')

        spots = pd.concat(all_tile_spots, ignore_index=True) if all_tile_spots else pd.DataFrame()
        accepted = pd.concat(all_tile_accepted, ignore_index=True) if all_tile_accepted else spots.copy()
        clear_region_memory(all_tile_spots, all_tile_accepted)

        cells = load_cell_features(r['features'], roi_yx=P.roi_yx)
        accepted = assign_spots_to_nearest_cell(accepted, cells)
        del cells
        if len(accepted):
            accepted['batch'] = batch
            accepted['region'] = region

        counts = build_cell_bit_count_table(accepted, region_markers)
        if len(counts):
            counts['total_spots'] = counts['decoded_spots']
            counts = counts[counts['total_spots'] >= P.min_spots_per_cell].reset_index(drop=True)
        else:
            counts = pd.DataFrame(columns=['cell', *region_markers, 'decoded_spots', 'decoded_exact_spots', 'decoded_tolerant_spots',
                                           'dominant_barcode', 'dominant_barcode_count', 'dominant_lnp_call', 'total_spots'])
        counts['batch'] = batch

        spots.to_csv(paths['all_candidates'], index=False)
        accepted.to_csv(paths['accepted'], index=False)
        counts.to_csv(paths['cell_counts'], index=False)

        tile_qa = pd.DataFrame(tile_qa_rows)
        n_hot = int(tile_qa['n_hot_pixels'].sum()) if len(tile_qa) and 'n_hot_pixels' in tile_qa else 0
        raw_bright_threshold = float(tile_qa['raw_bright_threshold'].replace([np.inf, -np.inf], np.nan).median()) if len(tile_qa) else np.nan
        raw_object_threshold = float(tile_qa['raw_object_threshold'].replace([np.inf, -np.inf], np.nan).median()) if len(tile_qa) else np.nan

        spot_qa = pd.DataFrame([dict(
            region=region,
            batch=batch,
            sample=sample,
            n_markers=len(region_markers),
            detection_source='full_barcode_multiscale_log_raw_rescue_manual_thresholds_tiled_checkpointed',
            n_region_tiles=len(tiles),
            region_tile_overlap=REGION_TILE_OVERLAP,
            min_on_snr_threshold=region_min_on_snr,
            log_sigmas=','.join(map(str, active_log_sigmas())),
            n_hot_pixels=n_hot,
            n_channel_candidates_total=int(tile_qa['n_channel_candidates_total'].sum()) if len(tile_qa) else 0,
            n_raw_bright_candidates=int(tile_qa['n_raw_bright_candidates'].sum()) if len(tile_qa) else 0,
            raw_bright_threshold=raw_bright_threshold,
            n_raw_object_candidates=int(tile_qa['n_raw_object_candidates'].sum()) if len(tile_qa) else 0,
            raw_object_threshold=raw_object_threshold,
            n_candidates=len(spots),
            n_spots=len(accepted),
            n_spots_assigned=int((accepted['cell'] > 0).sum()) if len(accepted) else 0,
            n_callable_cells=len(counts),
            n_exact=int(np.sum(accepted['barcode_match_status'] == 'exact')) if len(accepted) else 0,
            n_tolerant=int(np.sum(accepted['barcode_match_status'] == 'tolerant')) if len(accepted) else 0,
            median_min_on_snr=float(np.median(accepted['min_on_snr'])) if len(accepted) else np.nan,
            median_snr_margin=float(np.median(accepted['snr_margin'])) if len(accepted) else np.nan,
            frac_promiscuous=float(spots['promiscuous'].mean()) if len(spots) else np.nan,
        )])
        spot_qa.to_csv(paths['spot_qa'], index=False)
        log.info(f'{region}: wrote tiled checkpoint CSVs to {OUTDIR}')
        toc(t_region, f'{region} total')
    finally:
        clear_region_memory(stack, tissue_mask_full, all_tile_spots, all_tile_accepted, tile_qa_rows, spots, accepted, counts)


_REGION_UPSTREAM_HELPERS = (
    'open_stack', 'active_log_sigmas', 'build_hot_mask', 'detect_marker_log_candidates',
    'raw_rescue_candidate_tables', 'merge_marker_candidates', 'decode_spots_full_barcode',
    'build_cell_bit_count_table', 'threshold_for_marker',
    'selected_detection_params_for_region', 'detection_batch_for_region',
)
_missing_region_helpers = [name for name in _REGION_UPSTREAM_HELPERS if not callable(globals().get(name))]
if _missing_region_helpers:
    raise RuntimeError(
        'Section 6 is missing upstream helper definitions: '
        + ', '.join(_missing_region_helpers)
        + '. Restart the kernel and run the notebook cells in order through Section 6.'
    )


regions_to_run = set(RUN_REGIONS) if RUN_REGIONS is not None else None
all_spots_v4 = {}           # Kept for downstream compatibility; region tables are reloaded from disk.
all_accepted_spots_v4 = {}
all_cells_v4 = {}

for r in region_info:
    region = r['region']
    if regions_to_run is not None and region not in regions_to_run:
        log.info(f'{region}: not in RUN_REGIONS; skipping')
        continue
    if SKIP_COMPLETED_REGIONS and not FORCE_RERUN_REGION:
        checkpoint_complete, missing_required = region_checkpoint_status(region)
        if checkpoint_complete:
            log.info(f'{region}: required checkpoint CSVs already exist; skipping')
            continue
        missing_names = ', '.join(path.name for path in missing_required.values())
        log.info(f'{region}: missing required checkpoint CSVs ({missing_names}); running')
    process_region_spot_detection_v4(r)
    clear_region_memory()
    rss_gb = _current_rss_gb()
    if rss_gb is not None:
        log.info(f'{region}: resident memory after cleanup = {rss_gb:.2f} GB')

qa_image_parts = []
qa_spot_parts = []
for r in region_info:
    region = r['region']
    paths = region_checkpoint_paths(region)
    if paths['image_qa'].exists():
        qa_image_parts.append(pd.read_csv(paths['image_qa']))
    if paths['spot_qa'].exists():
        qa_spot_parts.append(pd.read_csv(paths['spot_qa']))

qa_per_image_v4 = pd.concat(qa_image_parts, ignore_index=False) if qa_image_parts else pd.DataFrame()
qa_spot_detection_v4 = pd.concat(qa_spot_parts, ignore_index=False) if qa_spot_parts else pd.DataFrame()
qa_per_image_v4.to_csv(OUTDIR / 'qa_per_image_v4.csv', index=False)
qa_spot_detection_v4.to_csv(OUTDIR / 'qa_spot_detection_v4.csv', index=False)
qa_spot_detection_v4


## 8. Combine region-level spot and cell-count tables


In [ ]:
combined_spots_v4 = []
combined_cells_v4 = []

spots_by_region = globals().get('all_accepted_spots_v4')
cells_by_region = globals().get('all_cells_v4')
if not isinstance(spots_by_region, dict):
    spots_by_region = {}
if not isinstance(cells_by_region, dict):
    cells_by_region = {}

for r in region_info:
    region = r['region']
    if region not in spots_by_region:
        spots_path = OUTDIR / f'spots_{region}_v4.csv'
        if spots_path.exists():
            spots_by_region[region] = pd.read_csv(spots_path)
    if region not in cells_by_region:
        cells_path = OUTDIR / f'cell_spot_counts_{region}_v4.csv'
        if cells_path.exists():
            cells_by_region[region] = pd.read_csv(cells_path)

for region in [r['region'] for r in region_info]:
    df = spots_by_region.get(region)
    if df is None:
        log.warning(f'{region}: missing accepted spot CSV; omit from combined spot table')
        continue
    tmp = df.copy()
    tmp['region'] = region
    cols = ['region'] + [c for c in tmp.columns if c != 'region']
    combined_spots_v4.append(tmp[cols])
for region in [r['region'] for r in region_info]:
    df = cells_by_region.get(region)
    if df is None:
        log.warning(f'{region}: missing cell-count CSV; omit from combined cell table')
        continue
    tmp = df.copy()
    tmp['region'] = region
    cols = ['region'] + [c for c in tmp.columns if c != 'region']
    combined_cells_v4.append(tmp[cols])

combined_spots_v4 = pd.concat(combined_spots_v4, ignore_index=True) if combined_spots_v4 else pd.DataFrame()
combined_cells_v4 = pd.concat(combined_cells_v4, ignore_index=True) if combined_cells_v4 else pd.DataFrame()
combined_spots_v4.to_csv(OUTDIR / 'spots_all_regions_v4.csv', index=False)
combined_cells_v4.to_csv(OUTDIR / 'cell_spot_counts_all_regions_v4.csv', index=False)

if combined_spots_v4.empty or 'lnp_call' not in combined_spots_v4.columns:
    summary_v4 = pd.DataFrame(columns=['region', 'lnp_call', 'n_spots'])
else:
    summary_v4 = (combined_spots_v4.groupby(['region', 'lnp_call']).size().rename('n_spots').reset_index())
summary_v4.to_csv(OUTDIR / 'decoded_spot_counts_by_lnp_v4.csv', index=False)
summary_v4


## 9. Display a representative spot/codebook comparison

This cell displays the comparison inline and does not write an image file.


In [ ]:
SHOWCASE_TARGET_REGION = 'S3_reg002'
SHOWCASE_TARGET_N_LNPS = 10
SHOWCASE_CROP_RADIUS = 25
SHOWCASE_DAPI_CANDIDATES = ['DAPI', 'Dapi', 'dapi']
SHOWCASE_EXCLUDED_LNP_CALLS = {'', 'no_barcode', 'unmapped', 'ambiguous_mixed'}


def find_dapi_marker(markers: List[str]) -> str:
    for name in SHOWCASE_DAPI_CANDIDATES:
        if name in markers:
            return name
    return ''


def draw_marker_border(ax, color: str):
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2.0)


def marker_bit_labels_for_code(code: str, markers: List[str]) -> pd.DataFrame:
    rows = []
    for marker, bit in zip(markers, code):
        rows.append({
            'marker': marker,
            'bit': int(bit),
            'bit_label': 'ON' if str(bit) == '1' else 'OFF',
        })
    return pd.DataFrame(rows)


def load_showcase_region_data(region: str):
    matches = [info for info in region_info if info['region'] == region]
    if not matches:
        raise ValueError(f'{region} was not found in region_info')
    info = matches[0]
    markers = info['markers']
    marker_to_idx = {m: i for i, m in enumerate(markers)}
    region_markers = [m for m in BARCODE_MARKERS if m in marker_to_idx]

    open_stack_fn = globals().get('open_stack')
    if open_stack_fn is None:
        import tifffile

        def open_stack_fn(path):
            return tifffile.imread(path)

    accepted_tables = globals().get('all_accepted_spots_v4', {})
    accepted = accepted_tables.get(region) if isinstance(accepted_tables, dict) else None
    if accepted is None:
        spots_path = OUTDIR / f'spots_{region}_v4.csv'
        if not spots_path.exists():
            raise FileNotFoundError(f'Accepted spot table not found: {spots_path}')
        accepted = pd.read_csv(spots_path)

    accepted = accepted.copy()
    if 'accepted' in accepted.columns:
        accepted = accepted[accepted['accepted']].copy()
    if accepted.empty:
        raise ValueError(f'{region} has no accepted decoded spots to showcase')

    cells = pd.read_csv(info['features'])
    if 'label' in cells.columns:
        cells = cells.rename(columns={'label': 'cell'})
    stack = open_stack_fn(info['image'])
    return info, stack, marker_to_idx, region_markers, accepted, cells


def choose_distinct_lnp_showcase_spots(accepted: pd.DataFrame, n_lnps: int) -> pd.DataFrame:
    y_col = 'i_global' if 'i_global' in accepted.columns else 'i'
    x_col = 'j_global' if 'j_global' in accepted.columns else 'j'
    required = {'lnp_call', 'called_code', y_col, x_col}
    missing = sorted(required - set(accepted.columns))
    if missing:
        raise ValueError(f'Accepted spot table is missing required columns: {missing}')

    ranked = accepted.copy()
    ranked['lnp_call'] = ranked['lnp_call'].fillna('').astype(str)
    ranked = ranked[~ranked['lnp_call'].isin(SHOWCASE_EXCLUDED_LNP_CALLS)].copy()
    if ranked.empty:
        raise ValueError('No mapped LNP calls are available for showcase after filtering no_barcode/unmapped/ambiguous calls')

    ranked['showcase_i_global'] = ranked[y_col].astype(float)
    ranked['showcase_j_global'] = ranked[x_col].astype(float)
    ranked['is_exact'] = ranked['barcode_match_status'].eq('exact') if 'barcode_match_status' in ranked.columns else False
    ranked['is_assigned'] = ranked['cell'].gt(0) if 'cell' in ranked.columns else False
    for col in ['snr_margin', 'min_on_snr']:
        if col not in ranked.columns:
            ranked[col] = np.nan
    ranked = ranked.sort_values(
        ['is_exact', 'is_assigned', 'snr_margin', 'min_on_snr'],
        ascending=[False, False, False, False],
        kind='mergesort',
    )
    return ranked.drop_duplicates('lnp_call', keep='first').head(n_lnps).reset_index(drop=True)


def marker_on_mask_from_spot_codes(spots: pd.DataFrame, marker: str, region_markers: List[str]) -> np.ndarray:
    if 'called_code' not in spots.columns or marker not in region_markers:
        return np.zeros(len(spots), dtype=bool)
    base_marker_on_mask_fn = globals().get('marker_on_mask_from_codes')
    if base_marker_on_mask_fn is not None:
        try:
            return np.asarray(base_marker_on_mask_fn(spots, marker), dtype=bool)
        except Exception:
            pass
    idx = region_markers.index(marker)
    codes = spots['called_code'].fillna('').astype(str).str.replace(r'\.0$', '', regex=True)
    codes = codes.str.pad(len(region_markers), side='right', fillchar='0')
    return codes.str[idx].eq('1').to_numpy()


def plot_spot_codebook_showcase(info: dict, rep: pd.Series, accepted: pd.DataFrame, cells: pd.DataFrame, stack, marker_to_idx, region_markers: List[str], rank_idx: int):
    dapi_marker = find_dapi_marker(info['markers'])
    code = str(rep['called_code'])
    h = int(stack.shape[1])
    w = int(stack.shape[2])
    rep_global_y = float(rep['showcase_i_global'])
    rep_global_x = float(rep['showcase_j_global'])

    rep_cell = int(rep.get('cell', 0)) if pd.notna(rep.get('cell', np.nan)) else 0
    cell_row = pd.Series(dtype=float)
    if rep_cell > 0 and 'cell' in cells.columns:
        hit = cells[cells['cell'].astype(int).eq(rep_cell)]
        if len(hit):
            cell_row = hit.iloc[0]

    if len(cell_row):
        cell_y = float(cell_row.get('y', rep_global_y))
        cell_x = float(cell_row.get('x', rep_global_x))
        major = float(cell_row.get('axis_major_length', SHOWCASE_CROP_RADIUS * 2))
        minor = float(cell_row.get('axis_minor_length', major))
        half_y = max(10.0, major / 2.0 + 5.0, abs(rep_global_y - cell_y) + 8.0)
        half_x = max(10.0, major / 2.0 + 5.0, abs(rep_global_x - cell_x) + 8.0)
        crop_center_y = cell_y
        crop_center_x = cell_x
    else:
        cell_y = rep_global_y
        cell_x = rep_global_x
        major = SHOWCASE_CROP_RADIUS * 1.8
        minor = major
        half_y = float(SHOWCASE_CROP_RADIUS)
        half_x = float(SHOWCASE_CROP_RADIUS)
        crop_center_y = rep_global_y
        crop_center_x = rep_global_x

    crop_y0 = max(0, int(np.floor(crop_center_y - half_y)))
    crop_x0 = max(0, int(np.floor(crop_center_x - half_x)))
    crop_y1 = min(h, int(np.ceil(crop_center_y + half_y + 1)))
    crop_x1 = min(w, int(np.ceil(crop_center_x + half_x + 1)))

    yy, xx = np.mgrid[crop_y0:crop_y1, crop_x0:crop_x1]
    ellipse_y = max(major / 2.0 + 3.0, abs(rep_global_y - cell_y) + 4.0, 6.0)
    ellipse_x = max(minor / 2.0 + 3.0, abs(rep_global_x - cell_x) + 4.0, 6.0)
    cell_mask = (((yy - cell_y) / ellipse_y) ** 2 + ((xx - cell_x) / ellipse_x) ** 2) <= 1.0
    spot_disk = ((yy - rep_global_y) ** 2 + (xx - rep_global_x) ** 2) <= 3.0 ** 2
    display_mask = cell_mask | spot_disk

    y_col = 'i_global' if 'i_global' in accepted.columns else 'i'
    x_col = 'j_global' if 'j_global' in accepted.columns else 'j'
    crop_spots = accepted.copy()
    crop_spots['global_i'] = crop_spots[y_col].astype(float)
    crop_spots['global_j'] = crop_spots[x_col].astype(float)
    crop_spots = crop_spots[
        (crop_spots['global_i'] >= crop_y0) & (crop_spots['global_i'] < crop_y1) &
        (crop_spots['global_j'] >= crop_x0) & (crop_spots['global_j'] < crop_x1)
    ].copy()
    if 'cell' in crop_spots.columns and rep_cell > 0:
        crop_spots = crop_spots[crop_spots['cell'].astype(int).eq(rep_cell)].copy()
    crop_spots['plot_i'] = crop_spots['global_i'] - crop_y0
    crop_spots['plot_j'] = crop_spots['global_j'] - crop_x0
    rep_y = rep_global_y - crop_y0
    rep_x = rep_global_x - crop_x0

    robust_limits_fn = globals().get('robust_limits')
    if robust_limits_fn is None:
        def robust_limits_fn(img: np.ndarray, p_low=1, p_high=99.85):
            vals = np.asarray(img)
            finite = vals[np.isfinite(vals)]
            if finite.size == 0:
                return 0.0, 1.0
            lo = float(np.percentile(finite, p_low))
            hi = float(np.percentile(finite, p_high))
            if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
                hi = lo + 1.0
            return lo, hi

    def normalize_for_overlay(img: np.ndarray, p_low=1, p_high=99.85) -> np.ndarray:
        lo, hi = robust_limits_fn(img, p_low=p_low, p_high=p_high)
        return np.clip((img.astype(float) - lo) / max(hi - lo, 1e-9), 0, 1)

    if dapi_marker:
        dapi_img = stack[marker_to_idx[dapi_marker], crop_y0:crop_y1, crop_x0:crop_x1]
        dapi_norm = normalize_for_overlay(dapi_img, p_low=1, p_high=99.7)
        dapi_cell = np.where(display_mask, dapi_norm, 0.0)
    else:
        dapi_cell = None

    bit_table = marker_bit_labels_for_code(code, region_markers)
    display_rows = []
    if dapi_marker:
        display_rows.append({'marker': dapi_marker, 'bit': np.nan, 'bit_label': 'DAPI'})
    display_rows.extend(bit_table.to_dict('records'))
    display_table = pd.DataFrame(display_rows)

    ncols = 4
    nrows = int(np.ceil(len(display_table) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.45 * ncols, 2.35 * nrows), squeeze=False)

    on_color = '#00b894'
    off_color = '#7f7f7f'
    off_ring = '#ff7a00'
    dapi_color = 'deepskyblue'
    arrow_color = 'yellow'
    on_cmap = 'Greens'
    off_cmap = 'Oranges'

    def draw_rep_arrow(ax, color: str):
        height = max(1, crop_y1 - crop_y0)
        width = max(1, crop_x1 - crop_x0)
        start_x = np.clip(rep_x - 0.38 * width, 1, width - 2)
        start_y = np.clip(rep_y - 0.38 * height, 1, height - 2)
        ax.annotate('', xy=(rep_x, rep_y), xytext=(start_x, start_y),
                    arrowprops=dict(arrowstyle='-|>', color='black', lw=4.0, shrinkA=0, shrinkB=1), zorder=30)
        ax.annotate('', xy=(rep_x, rep_y), xytext=(start_x, start_y),
                    arrowprops=dict(arrowstyle='-|>', color=color, lw=2.1, shrinkA=0, shrinkB=1), zorder=31)

    for ax, (_, row) in zip(axes.ravel(), display_table.iterrows()):
        marker = row['marker']
        img = stack[marker_to_idx[marker], crop_y0:crop_y1, crop_x0:crop_x1]

        if marker == dapi_marker or dapi_cell is None:
            if dapi_cell is not None:
                ax.imshow(dapi_cell, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
            else:
                lo, hi = robust_limits_fn(img, p_low=1, p_high=99.85)
                ax.imshow(img, cmap='gray', vmin=lo, vmax=hi, interpolation='nearest')
        else:
            bit = int(row['bit'])
            marker_norm = normalize_for_overlay(img, p_low=5, p_high=99.95)
            marker_norm = np.where(display_mask, marker_norm, 0.0)
            marker_alpha = np.where(marker_norm > 0.08, np.clip(0.25 + marker_norm * 0.85, 0, 0.95), 0.0)
            ax.imshow(dapi_cell * 0.55, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
            ax.imshow(marker_norm, cmap=on_cmap if bit == 1 else off_cmap,
                      vmin=0, vmax=1, alpha=marker_alpha, interpolation='nearest')

        if marker == dapi_marker:
            draw_rep_arrow(ax, arrow_color)
            draw_marker_border(ax, dapi_color)
            ax.set_title(f'{marker} assigned cell  (n={len(crop_spots)})', color=dapi_color, fontsize=9, fontweight='bold')
        else:
            bit = int(row['bit'])
            mask = marker_on_mask_from_spot_codes(crop_spots, marker, region_markers)
            marker_spots = crop_spots.iloc[np.where(mask)[0]] if mask.size else crop_spots.iloc[[]]
            draw_rep_arrow(ax, on_color if bit == 1 else off_ring)
            ax.set_title(
                f"{marker}  {row['bit_label']} on cell  (n={len(marker_spots)})",
                color=on_color if bit == 1 else off_color,
                fontsize=9,
                fontweight='bold',
            )
            draw_marker_border(ax, on_color if bit == 1 else off_color)

        ax.set_xticks([])
        ax.set_yticks([])

    for ax in axes.ravel()[len(display_table):]:
        ax.axis('off')

    decoded_on = ', '.join([m for m, b in zip(region_markers, code) if b == '1'])
    match_dist = rep.get('barcode_match_distance', np.nan)
    match_dist_text = 'NA' if pd.isna(match_dist) else str(int(match_dist))
    subtitle = (
        f"{rank_idx}. {info['region']}  |  {rep.get('lnp_call', 'unknown')}  |  assigned cell {rep_cell or 'NA'}  |  code {code}  |  "
        f"{rep.get('barcode_match_status', 'unknown')} (Hamming {match_dist_text})\n"
        f"Representative spot ON markers: {decoded_on}\n"
        f"cell accepted spots={len(crop_spots)},  min_on_snr={float(rep.get('min_on_snr', np.nan)):.2f},  "
        f"snr_margin={float(rep.get('snr_margin', np.nan)):.2f}"
    )
    fig.suptitle(
        'Decoded barcode showcase: assigned-cell DAPI crop with marker overlay and arrow to spot\n'
        f'{subtitle}',
        fontsize=10.5,
        fontweight='bold',
        y=1.01,
    )
    plt.tight_layout()
    plt.show()


def run_codebook_showcase():
    info, stack, marker_to_idx, region_markers, accepted, cells = load_showcase_region_data(SHOWCASE_TARGET_REGION)
    reps = choose_distinct_lnp_showcase_spots(accepted, SHOWCASE_TARGET_N_LNPS)
    log.info(f"Codebook showcase region: {SHOWCASE_TARGET_REGION}; displaying {len(reps)} distinct LNP calls")

    rows = []
    for rank_idx, (_, rep) in enumerate(reps.iterrows(), start=1):
        plot_spot_codebook_showcase(info, rep, accepted, cells, stack, marker_to_idx, region_markers, rank_idx)
        rows.append({
            'region': SHOWCASE_TARGET_REGION,
            'showcase_rank': rank_idx,
            'lnp_call': rep.get('lnp_call', ''),
            'called_code': rep.get('called_code', ''),
            'barcode_match_status': rep.get('barcode_match_status', ''),
            'barcode_match_distance': rep.get('barcode_match_distance', np.nan),
            'min_on_snr': rep.get('min_on_snr', np.nan),
            'snr_margin': rep.get('snr_margin', np.nan),
            'i_global': rep.get('showcase_i_global', np.nan),
            'j_global': rep.get('showcase_j_global', np.nan),
        })
    return pd.DataFrame(rows)


codebook_showcase_summary_v4 = run_codebook_showcase()
codebook_showcase_summary_v4


## 10. Build the final cell-level analysis tables


In [ ]:
def hamming_distance(a: str, b: str) -> int:
    if len(a) != len(b):
        return max(len(a), len(b))
    return sum(x != y for x, y in zip(a, b))


def tolerant_lnp_match(barcode: str, total_spots: int) -> dict:
    if total_spots == 0:
        return dict(lnp_call='no_barcode', barcode_in_library=False,
                    barcode_match_distance=np.nan, barcode_match_status='no_barcode',
                    barcode_excluded=True)
    if not BARCODE_LIBRARY:
        return dict(lnp_call='unmapped', barcode_in_library=False,
                    barcode_match_distance=np.nan, barcode_match_status='unmapped',
                    barcode_excluded=False)

    distances = [(lib_bc, name, hamming_distance(barcode, lib_bc))
                 for lib_bc, name in BARCODE_LIBRARY.items()]
    min_dist = min(d for _, _, d in distances)
    candidates = [(lib_bc, name, d) for lib_bc, name, d in distances
                  if d == min_dist and d <= P.barcode_max_hamming_distance]

    if len(candidates) == 1:
        lib_bc, name, dist = candidates[0]
        return dict(lnp_call=name, barcode_in_library=(dist == 0),
                    barcode_match_distance=dist,
                    barcode_match_status='exact' if dist == 0 else 'tolerant',
                    barcode_excluded=False)
    if len(candidates) > 1:
        return dict(lnp_call='ambiguous_mixed', barcode_in_library=False,
                    barcode_match_distance=min_dist, barcode_match_status='ambiguous_mixed',
                    barcode_excluded=True)
    return dict(lnp_call='unmapped', barcode_in_library=False,
                barcode_match_distance=min_dist, barcode_match_status='unmapped',
                barcode_excluded=False)


def build_cell_analysis_table_for_region(region: str, info: dict, spots: pd.DataFrame) -> pd.DataFrame:
    cells = pd.read_csv(info['features'])
    if 'label' not in cells.columns or 'x' not in cells.columns or 'y' not in cells.columns:
        raise ValueError(f"{info['features']} must contain label, x, and y columns")

    cells = cells.copy()
    cells.insert(0, 'region', region)
    cells = cells.rename(columns={'label': 'cell'})

    assigned = spots[spots['cell'] > 0].copy() if len(spots) else pd.DataFrame()
    if len(assigned):
        def normalize_called_code(code: object) -> str | None:
            if pd.isna(code):
                return None
            if isinstance(code, (int, np.integer)):
                text = str(int(code))
            elif isinstance(code, float):
                if not float(code).is_integer():
                    return None
                text = str(int(code))
            else:
                text = str(code).strip()
                if text.endswith('.0') and text[:-2].isdigit():
                    text = text[:-2]
            if not text or any(ch not in '01' for ch in text):
                return None
            return text.zfill(len(BARCODE_MARKERS))

        assigned['called_code'] = assigned['called_code'].map(normalize_called_code)
        assigned = assigned[assigned['called_code'].notna()].copy()
    if len(assigned):
        bit_matrix = np.array([[int(ch) for ch in code] for code in assigned['called_code']], dtype=int)
        counts = pd.DataFrame(bit_matrix, columns=BARCODE_MARKERS, index=assigned.index)
        counts['cell'] = assigned['cell'].to_numpy(int)
        counts = counts.groupby('cell')[BARCODE_MARKERS].sum().reset_index()
        counts.columns = ['cell', *[f'spot_{m}' for m in BARCODE_MARKERS]]

        decoded_summary = assigned.groupby('cell').agg(
            decoded_spots=('called_code', 'size'),
            decoded_exact_spots=('barcode_match_status', lambda s: int(np.sum(s == 'exact'))),
            decoded_tolerant_spots=('barcode_match_status', lambda s: int(np.sum(s == 'tolerant'))),
        ).reset_index()

        per_bc = (assigned.groupby(['cell', 'called_code']).size().rename('n').reset_index()
                  .sort_values(['cell', 'n', 'called_code'], ascending=[True, False, True]))
        dominant_bc = per_bc.drop_duplicates('cell').rename(columns={'called_code': 'dominant_decoded_barcode', 'n': 'dominant_decoded_barcode_count'})
    else:
        counts = pd.DataFrame({'cell': cells['cell']})
        for marker in BARCODE_MARKERS:
            counts[f'spot_{marker}'] = 0
        decoded_summary = pd.DataFrame({'cell': cells['cell'], 'decoded_spots': 0, 'decoded_exact_spots': 0, 'decoded_tolerant_spots': 0})
        dominant_bc = pd.DataFrame({'cell': cells['cell'], 'dominant_decoded_barcode': '', 'dominant_decoded_barcode_count': 0})

    out = cells.merge(counts, on='cell', how='left')
    out = out.merge(decoded_summary, on='cell', how='left')
    out = out.merge(dominant_bc[['cell', 'dominant_decoded_barcode', 'dominant_decoded_barcode_count']], on='cell', how='left')

    spot_cols = [f'spot_{m}' for m in BARCODE_MARKERS]
    fill_zero_cols = spot_cols + ['decoded_spots', 'decoded_exact_spots', 'decoded_tolerant_spots', 'dominant_decoded_barcode_count']
    for col in fill_zero_cols:
        if col not in out.columns:
            out[col] = 0
        out[col] = out[col].fillna(0)
    for col in spot_cols + ['decoded_spots', 'decoded_exact_spots', 'decoded_tolerant_spots', 'dominant_decoded_barcode_count']:
        out[col] = out[col].astype(int)
    out['dominant_decoded_barcode'] = out['dominant_decoded_barcode'].fillna('')

    bit_cols = []
    for marker in BARCODE_MARKERS:
        bit_col = f'bit_{marker}'
        spot_col = f'spot_{marker}'
        out[bit_col] = (out[spot_col] >= P.min_spots_for_bit).astype(int)
        bit_cols.append(bit_col)

    out['barcode'] = out[bit_cols].astype(str).agg(''.join, axis=1)
    out['total_barcode_spots'] = out['decoded_spots']
    out['n_positive_bits'] = out[bit_cols].sum(axis=1)

    spot_matrix = out[spot_cols].to_numpy(int)
    if len(out):
        dominant_idx = np.argmax(spot_matrix, axis=1)
        dominant_counts = spot_matrix[np.arange(len(out)), dominant_idx]
        out['dominant_barcode_marker'] = [BARCODE_MARKERS[i] if total > 0 else 'none'
                                          for i, total in zip(dominant_idx, out['total_barcode_spots'])]
        out['dominant_barcode_count'] = dominant_counts
        out['dominant_barcode_fraction'] = np.where(
            out['total_barcode_spots'] > 0,
            out['dominant_barcode_count'] / out['total_barcode_spots'],
            0.0,
        )
    else:
        out['dominant_barcode_marker'] = []
        out['dominant_barcode_count'] = []
        out['dominant_barcode_fraction'] = []

    positive_spot_sum = np.zeros(len(out), dtype=int)
    for marker in BARCODE_MARKERS:
        positive_spot_sum += out[f'spot_{marker}'].where(out[f'bit_{marker}'] == 1, 0).to_numpy(int)
    out['barcode_confidence'] = np.where(
        out['total_barcode_spots'] > 0,
        positive_spot_sum / (out['total_barcode_spots'] * max(P.expected_on_bits, 1)),
        0.0,
    )

    match_rows = [tolerant_lnp_match(bc, int(total))
                  for bc, total in zip(out['barcode'], out['total_barcode_spots'])]
    match_df = pd.DataFrame(match_rows, index=out.index)
    for col in match_df.columns:
        out[col] = match_df[col]
    out['lnp_positive'] = (
        (~out['barcode_excluded'])
        & ~out['lnp_call'].isin(['no_barcode', 'unmapped', 'ambiguous_mixed'])
    )

    front = [
        'region', 'cell', 'x', 'y', 'area', 'eccentricity',
        'barcode', 'lnp_call', 'lnp_positive', 'barcode_in_library',
        'barcode_match_distance', 'barcode_match_status', 'barcode_excluded',
        'total_barcode_spots', 'decoded_exact_spots', 'decoded_tolerant_spots',
        'n_positive_bits', 'dominant_decoded_barcode', 'dominant_decoded_barcode_count',
        'dominant_barcode_marker', 'dominant_barcode_count', 'dominant_barcode_fraction',
        'barcode_confidence',
    ]
    front = [c for c in front if c in out.columns]
    rest = [c for c in out.columns if c not in front]
    return out[front + rest]


cell_analysis_tables_v4 = []
spots_by_region = globals().get('all_accepted_spots_v4')
for info in region_info:
    region = info['region']
    region_spots = pd.DataFrame()
    if isinstance(spots_by_region, dict):
        region_spots = spots_by_region.get(region, pd.DataFrame())
    if region_spots.empty:
        spots_path = OUTDIR / f'spots_{region}_v4.csv'
        if spots_path.exists():
            region_spots = pd.read_csv(spots_path)
    table = build_cell_analysis_table_for_region(region, info, region_spots)
    table.to_csv(OUTDIR / f'cell_analysis_table_{region}_v4.csv', index=False)
    cell_analysis_tables_v4.append(table)
    log.info(f"{region}: wrote cell_analysis_table_{region}_v4.csv with {len(table)} cells")

cell_analysis_table_v4 = pd.concat(cell_analysis_tables_v4, ignore_index=True)
cell_analysis_table_v4.to_csv(OUTDIR / 'cell_analysis_table_all_regions_v4.csv', index=False)
log.info(f"Wrote {OUTDIR / 'cell_analysis_table_all_regions_v4.csv'} with {len(cell_analysis_table_v4)} cells")

cell_analysis_summary_v4 = (cell_analysis_table_v4
                            .groupby(['region', 'lnp_call'], dropna=False)
                            .size()
                            .reset_index(name='n_cells'))
cell_analysis_summary_v4.to_csv(OUTDIR / 'cell_analysis_summary_by_lnp_call_v4.csv', index=False)

print('Full 12-bit barcode summary (v4):')
display(cell_analysis_summary_v4)


## Generated outputs

The workflow writes CSV tables and resumable detection checkpoints under the publication data folder. It does not save manuscript image panels. Cell segmentation itself was performed by the previously published segmentation workflow and is not repeated here; the supplied feature CSVs provide the cell centroids used for spot assignment.
